<a href="https://colab.research.google.com/github/SAINIDHI2005/IDS_GNN_Repo/blob/main/graphSAGE/KNN_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATASET_PATH = "/content/drive/MyDrive/CIC_IoT_2023/Host_flow_temporal"

In [3]:
import os

print(os.listdir(DATASET_PATH))

['BENIGN.csv', 'DDoS.csv', 'Spoofing.csv', 'Recon.csv', 'Mirai.csv', 'DoS.csv']


In [4]:
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.7 MB/s eta 0:00:00


In [5]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

import torch
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv

from sklearn.neighbors import kneighbors_graph

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

import matplotlib.pyplot as plt
import seaborn as sns

In [27]:
ATTACK_FOLDERS = [

    "DDOS",
    "DOS",
    "MIRAI",
    "RECON",
    "SPOOFING"
]

In [28]:
benign_df = pd.read_csv(
    os.path.join(
        DATASET_PATH,
        "BENIGN.csv"
    )
)

benign_df = benign_df.sample(
    n=200000,
    random_state=42
)

benign_df["label"] = 0

print(len(benign_df))

200000


In [29]:
ATTACK_FILES = [
    "DDoS",
    "DoS",
    "Mirai",
    "Recon",
    "Spoofing"
]

attack_dfs = []

for attack in ATTACK_FILES:

    attack_df = pd.read_csv(
        os.path.join(
            DATASET_PATH,
            f"{attack}.csv"
        )
    )

    attack_df = attack_df.sample(
        n=40000,
        random_state=42
    )

    attack_df["label"] = 1

    attack_dfs.append(attack_df)

    print(attack, len(attack_df))

DDoS 40000
DoS 40000
Mirai 40000
Recon 40000
Spoofing 40000


In [30]:
malicious_df = pd.concat(
    attack_dfs,
    ignore_index=True
)

print(len(malicious_df))

200000


In [31]:
df = pd.concat(

    [benign_df, malicious_df],

    ignore_index=True
)

df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(df.shape)

(400000, 87)


In [32]:
print(df["label"].value_counts())

label
0    200000
1    200000
Name: count, dtype: int64


In [33]:
df.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

df.fillna(0, inplace=True)

print(df.shape)

(400000, 87)


In [34]:
FEATURES_30 = [
    # paste your exact 30 features here
     "bidirectional_mean_ps",
    "dst2src_syn_packets",
    "src2dst_min_ps",
    "ip_version",
    "src2dst_max_piat_ms",
    "dst2src_stddev_piat_ms",
    "bidirectional_max_piat_ms",
    "bidirectional_rst_packets",
    "src2dst_max_ps",
    "src2dst_duration_ms",
    "src2dst_bytes",
    "src2dst_ece_packets",
    "bidirectional_max_ps",
    "dst2src_bytes",
    "bidirectional_bytes",
    "dst2src_max_ps",
    "bidirectional_min_ps",
    "bidirectional_syn_packets",
    "dst2src_stddev_ps",
    "bidirectional_ack_packets",
    "protocol",
    "dst_port",
    "bidirectional_mean_piat_ms",
    "dst2src_duration_ms",
    "src2dst_mean_ps",
    "src2dst_min_piat_ms",
    "bidirectional_stddev_ps",
    "src2dst_stddev_piat_ms",
    "bidirectional_cwr_packets",
    "src2dst_psh_packets"
]

X = df[FEATURES_30]

y = df["label"]

In [35]:
print("Number of features:", len(FEATURES_30))

Number of features: 30


In [36]:
missing = [f for f in FEATURES_30 if f not in df.columns]

print("Missing features:", missing)

Missing features: []


In [37]:
print(X.shape)
print(y.shape)

(400000, 30)
(400000,)


In [38]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [39]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(
    X_train
)

X_val = scaler.transform(
    X_val
)


In [40]:
GRAPH_SIZE = 1000

K = 15

In [41]:
def create_graph_dataset(X, y):

    graphs = []

    num_graphs = len(X) // GRAPH_SIZE

    for i in range(num_graphs):

        start = i * GRAPH_SIZE

        end = start + GRAPH_SIZE

        X_chunk = X[start:end]

        y_chunk = y.iloc[start:end]

        adjacency = kneighbors_graph(

            X_chunk,

            n_neighbors=K,

            mode='connectivity',

            include_self=False
        )

        edge_index = np.array(
            adjacency.nonzero()
        )

        edge_index = torch.tensor(
            edge_index,
            dtype=torch.long
        )

        x_tensor = torch.tensor(
            X_chunk,
            dtype=torch.float
        )

        y_tensor = torch.tensor(
            y_chunk.values,
            dtype=torch.long
        )

        graph = Data(

            x=x_tensor,

            edge_index=edge_index,

            y=y_tensor
        )

        graphs.append(graph)

    return graphs

In [42]:
train_graphs = create_graph_dataset(
    X_train,
    y_train
)

val_graphs = create_graph_dataset(
    X_val,
    y_val
)


In [43]:
train_loader = DataLoader(
    train_graphs,
    batch_size=1,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=1,
    shuffle=False
)

In [44]:
class GraphSAGE(torch.nn.Module):

    def __init__(

        self,

        in_channels,

        hidden_channels,

        num_classes
    ):

        super().__init__()

        self.conv1 = SAGEConv(
            in_channels,
            hidden_channels
        )

        self.bn1 = torch.nn.BatchNorm1d(
            hidden_channels
        )

        self.conv2 = SAGEConv(
            hidden_channels,
            hidden_channels
        )

        self.bn2 = torch.nn.BatchNorm1d(
            hidden_channels
        )

        self.linear = torch.nn.Linear(
            hidden_channels,
            num_classes
        )

    def forward(
        self,
        x,
        edge_index
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = self.bn1(x)

        x = F.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        x = self.bn2(x)

        x = F.relu(x)

        x = self.linear(x)

        return x

In [45]:
device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'cpu'
)

print(f"Device Selected: {device}")

model = GraphSAGE(

    in_channels=X_train.shape[1],

    hidden_channels=256,

    num_classes=2

).to(device)

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=0.001
)

criterion = torch.nn.CrossEntropyLoss()

Device Selected: cuda


In [46]:
def validate():

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for graph in val_loader:

            graph = graph.to(device)

            out = model(
                graph.x,
                graph.edge_index
            )

            pred = out.argmax(dim=1)

            correct += (pred == graph.y).sum().item()
            total += graph.y.size(0)

    return correct / total

In [47]:
import os
import time
import torch
import joblib

SAVE_DIR = "/content/drive/MyDrive/CIC_IoT_2023/Saved_Model"
os.makedirs(SAVE_DIR, exist_ok=True)

best_val_acc = 0.0
best_epoch = 0

start_time = time.time()

EPOCHS = 100

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for graph in train_loader:

        graph = graph.to(device)

        optimizer.zero_grad()

        out = model(
            graph.x,
            graph.edge_index
        )

        loss = criterion(
            out,
            graph.y
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for graph in val_loader:

            graph = graph.to(device)

            out = model(
                graph.x,
                graph.edge_index
            )

            pred = out.argmax(dim=1)

            correct += (pred == graph.y).sum().item()
            total += graph.y.size(0)

    val_acc = correct / total

    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch + 1

        torch.save(
            model.state_dict(),
            os.path.join(
                SAVE_DIR,
                "KNN_graphSAGE.pth"
            )
        )

        joblib.dump(
            scaler,
            os.path.join(
                SAVE_DIR,
                "KNN_scaler.pkl"
            )
        )

    avg_loss = total_loss / len(train_loader)
    print(
        f"Epoch {epoch+1:03d}"
        f" | Loss {avg_loss:.4f}"
        f" | Validation {val_acc:.4f}"
    )

end_time = time.time()

training_time = end_time - start_time

print("\n" + "="*60)

print("Training Finished Successfully!")

print(f"Best Validation Accuracy : {best_val_acc:.4f}")
print(f"Best Epoch               : {best_epoch}")
print(f"Training Time            : {training_time:.2f} seconds")
print(f"Training Time            : {training_time/60:.2f} minutes")

print(f"\nBest model saved to:")
print(os.path.join(SAVE_DIR, "KNN_graphSAGE.pth"))

print(f"\nScaler saved to:")
print(os.path.join(SAVE_DIR, "KNN_scaler.pkl"))

print("="*60)

Epoch 001 | Loss 0.4172 | Validation 0.8215
Epoch 002 | Loss 0.3689 | Validation 0.8237
Epoch 003 | Loss 0.3501 | Validation 0.8293
Epoch 004 | Loss 0.3355 | Validation 0.8542
Epoch 005 | Loss 0.3282 | Validation 0.8555
Epoch 006 | Loss 0.3189 | Validation 0.8544
Epoch 007 | Loss 0.3105 | Validation 0.8497
Epoch 008 | Loss 0.3056 | Validation 0.8608
Epoch 009 | Loss 0.2995 | Validation 0.8532
Epoch 010 | Loss 0.2962 | Validation 0.8645
Epoch 011 | Loss 0.2899 | Validation 0.8625
Epoch 012 | Loss 0.2865 | Validation 0.8667
Epoch 013 | Loss 0.2849 | Validation 0.8634
Epoch 014 | Loss 0.2812 | Validation 0.8839
Epoch 015 | Loss 0.2784 | Validation 0.8831
Epoch 016 | Loss 0.2761 | Validation 0.8844
Epoch 017 | Loss 0.2737 | Validation 0.8862
Epoch 018 | Loss 0.2705 | Validation 0.8840
Epoch 019 | Loss 0.2690 | Validation 0.8884
Epoch 020 | Loss 0.2672 | Validation 0.8891
Epoch 021 | Loss 0.2657 | Validation 0.8875
Epoch 022 | Loss 0.2640 | Validation 0.8910
Epoch 023 | Loss 0.2622 | Valida